# 01 · Exploración y Preparación de Datos — Andina Crédito

**Objetivo:** modelo de probabilidad de *default* (mora 90+ días a 12 meses).

Contenido:
1. Exploración estructural.
2. Target.
3. Calidad de datos (problemas + decisiones).
4. Limpieza (`src/prepare.py`).
5. **Análisis de correlaciones** (tabla + matriz interactiva + ranking).
6. **Señal predictiva del Top 10** — y la **fuga de información** que aparece en el ranking.
7. **Análisis del pricing** (`tasa_interes_anual`).
8. Análisis temporal (define la validación).
9. Traducción a la política de aprobación (umbral económico).

> Gráficos **interactivos con Plotly** · paleta corporativa **Banco BICE**.
> Limpieza centralizada en `src/prepare.py` (fit/transform, sin *data leakage*).
> **Regla de trabajo:** ninguna conclusión sin un gráfico que permita verificarla.

In [ ]:
import sys, os
sys.path.append(os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from sklearn.metrics import roc_auc_score

pio.renderers.default = 'notebook'

# ---- Paleta corporativa Banco BICE ----
BICE_AZUL       = '#0E162A'   # azul marino (principal)
BICE_AZUL_MED   = '#2E536D'   # azul medio
BICE_AZUL_CLARO = '#A9BDDF'   # azul claro
BICE_ACENTO     = '#CE894D'   # terracota (acento / destacado)
BICE_PETROLEO   = '#062C33'   # azul petróleo (contraste)
BICE_ALERTA     = '#C00000'   # rojo (alertas / hallazgos críticos)
BICE_OK         = '#2E7D32'   # verde (validaciones correctas)
PALETA      = [BICE_AZUL, BICE_ACENTO]                 # 2 categorías
ESCALA_BICE = ['#A9BDDF', '#2E536D', '#0E162A']        # escala continua claro→oscuro

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

## 1. Exploración estructural

In [ ]:
print('Columnas solo en train:', set(train.columns) - set(test.columns))
train.dtypes

In [ ]:
train.head()

In [ ]:
# Dimensiones y ventanas temporales: el desfase train→test define toda la validación
for df, n in [(train, 'train'), (test, 'test')]:
    f = pd.to_datetime(df['fecha_solicitud'])
    print(f'{n:>5}: {df.shape[0]:,} filas × {df.shape[1]} columnas  |  '
          f'{f.min().date()} → {f.max().date()}')

**Hallazgo:** train y test **no se solapan en el tiempo** (train cierra en 2025-02, test cubre
2025-02 → 2025-06). El ejercicio es explícitamente *out-of-time*, no una muestra aleatoria de
la misma población. Esto condiciona la validación (sección 8) y obliga a revisar deriva.

## 2. Target: `default_12m`

In [ ]:
dist = train['default_12m'].value_counts().sort_index()
tasa = train['default_12m'].mean()*100
print(dist); print(f"\nTasa de default: {tasa:.2f}%")

fig = px.bar(x=['No default (0)','Default (1)'], y=dist.values, text=dist.values,
             color=['No default (0)','Default (1)'], color_discrete_sequence=PALETA,
             title=f'Distribución del target · tasa de default = {tasa:.2f}%')
fig.update_layout(showlegend=False, xaxis_title='', yaxis_title='N° solicitudes')
fig.update_traces(textposition='outside'); fig.show()

**Hallazgo:** ~9,9% de default → desbalance. Usaremos **AUC-ROC, KS, PR-AUC** (no *accuracy*).

**Advertencia que se desarrolla en la sección 8:** este 9,9% es un **promedio de un período en
el que la tasa se duplicó**. No es la tasa esperada para el período de test.

## 3. Calidad de datos — problemas detectados
### 3.1 Valores nulos

In [ ]:
nulos = pd.DataFrame({'nulos_train': train.isnull().sum(),
    'pct_train': (train.isnull().mean()*100).round(1),
    'pct_test': (test.isnull().mean()*100).round(1)})
nulos = nulos[nulos['nulos_train'] > 0]; print(nulos)

fig = go.Figure()
fig.add_bar(x=nulos.index, y=nulos['pct_train'], name='train', marker_color=BICE_AZUL_MED,
            text=nulos['pct_train'], texttemplate='%{text}%', textposition='outside')
fig.add_bar(x=nulos.index, y=nulos['pct_test'], name='test', marker_color=BICE_ACENTO,
            text=nulos['pct_test'], texttemplate='%{text}%', textposition='outside')
fig.update_layout(title='% de valores nulos por columna — train vs test',
                  xaxis_title='', yaxis_title='% nulos', barmode='group')
fig.show()

**Hallazgo:** `ingreso_declarado` (~18%) y `antiguedad_laboral_meses` (~16%).
La tasa es prácticamente **idéntica en train y test** → el mecanismo de falta es estable en
el tiempo, así que una imputación ajustada en train transfiere bien.

#### ¿El faltante es informativo?
Antes de imputar hay que saber si "no informó ingreso" predice default. Si la tasa difiere
entre filas con y sin dato, la bandera de faltante es una *feature* en sí misma.

In [ ]:
filas = []
for c in ['ingreso_declarado', 'antiguedad_laboral_meses']:
    m = train[c].isna()
    filas.append({'variable': c, 'estado': 'faltante', 'tasa': train.loc[m, 'default_12m'].mean()})
    filas.append({'variable': c, 'estado': 'presente', 'tasa': train.loc[~m, 'default_12m'].mean()})
info_falt = pd.DataFrame(filas)

fig = px.bar(info_falt, x='variable', y='tasa', color='estado', barmode='group',
             color_discrete_sequence=[BICE_ACENTO, BICE_AZUL], text='tasa',
             title='Tasa de default según si el dato está presente o faltante')
fig.add_hline(y=train['default_12m'].mean(), line_dash='dot', line_color=BICE_PETROLEO,
              annotation_text=f"tasa global {train['default_12m'].mean():.1%}")
fig.update_traces(texttemplate='%{text:.2%}', textposition='outside')
fig.update_layout(xaxis_title='', yaxis_title='tasa de default', yaxis_tickformat='.1%')
fig.show()
print(info_falt.assign(tasa=lambda d: (d.tasa*100).round(2)).to_string(index=False))

**Decisión:** imputar mediana (calculada **solo en train**) + **bandera de faltante**.
La diferencia en tasa es leve pero la bandera no cuesta nada y preserva la información.

### 3.2 Rangos imposibles — edad

In [ ]:
fig = px.histogram(train, x='edad', nbins=60, color_discrete_sequence=[BICE_AZUL],
                   title='Distribución de edad (con outliers)')
fig.add_vline(x=100, line_dash='dash', line_color=BICE_ACENTO,
              annotation_text='límite 100 años', annotation_position='top')
fig.update_layout(xaxis_title='edad', yaxis_title='frecuencia', bargap=0.02); fig.show()
print('Edad > 100:', (train['edad'] > 100).sum(), 'filas (hasta', int(train['edad'].max()), 'años)')
print('Edad > 100 en test:', (test['edad'] > 100).sum(), '← el problema es exclusivo de train')

**Hallazgo:** 70 filas con edad ≥ 100 (hasta 133). **Decisión:** → nulo → imputar + `flag_edad_invalida`.

### 3.3 Rangos imposibles — antigüedad laboral mayor que la vida laboral

El mismo tipo de problema aparece de forma **cruzada**: registros cuya antigüedad laboral
excede los meses que esa persona pudo haber trabajado desde los 18 años. No se detecta
mirando la variable sola, solo al contrastarla contra la edad.

In [ ]:
tope = (train['edad'] - 18) * 12
invalida = (train['antiguedad_laboral_meses'] > tope)
m = train.sample(min(9000, len(train)), random_state=1)

fig = go.Figure()
fig.add_scatter(x=m['edad'], y=m['antiguedad_laboral_meses'], mode='markers',
                name='registros (muestra)', marker=dict(color=BICE_AZUL_CLARO, size=4, opacity=0.5))
fig.add_scatter(x=train.loc[invalida.fillna(False), 'edad'],
                y=train.loc[invalida.fillna(False), 'antiguedad_laboral_meses'],
                mode='markers', name='imposibles',
                marker=dict(color=BICE_ALERTA, size=9, symbol='x'))
xs = np.arange(19, int(train['edad'].max()) + 1)
fig.add_scatter(x=xs, y=(xs - 18) * 12, mode='lines', name='límite (edad−18)×12',
                line=dict(color=BICE_ACENTO, dash='dash', width=2))
fig.update_layout(title='Antigüedad laboral vs edad — los puntos sobre la recta son imposibles',
                  xaxis_title='edad (años)', yaxis_title='antigüedad laboral (meses)', height=460)
fig.show()

print('Registros con antigüedad > vida laboral posible:', int(invalida.fillna(False).sum()))
print('Antigüedad laboral > 600 meses (50 años):', int((train['antiguedad_laboral_meses'] > 600).sum()),
      '| máximo:', int(train['antiguedad_laboral_meses'].max()), 'meses')

**Hallazgo:** 46 registros con antigüedad laboral lógicamente imposible (el máximo son 936
meses = 78 años trabajando). Varios coinciden con las edades imposibles de 3.2, pero no todos.
**Decisión:** → nulo → imputar + `flag_antiguedad_invalida`.

### 3.4 Inconsistencia de unidades — ingreso (hallazgo principal de calidad)

In [ ]:
bajos = train[train['ingreso_declarado'] < 50000]['ingreso_declarado']
print(f"Filas con ingreso < 50.000 CLP: {len(bajos):,} (~{len(bajos)/len(train)*100:.0f}% del total)")
print(f"Máx del grupo bajo: {bajos.max():,.0f}")
ing = train['ingreso_declarado'].dropna()
print(f"Valores entre 5.014 y 280.000: {((ing>5014)&(ing<280000)).sum()}  <- 0 = gap limpio")

fig = px.histogram(x=np.log10(ing), nbins=90, color_discrete_sequence=[BICE_AZUL],
                   title='Distribución de ingreso_declarado en log10 — bimodalidad por unidad de medida')
fig.add_vline(x=np.log10(50000), line_dash='dash', line_color=BICE_ACENTO,
              annotation_text='corte 50.000', annotation_position='top')
fig.update_layout(xaxis_title='log10(ingreso declarado)', yaxis_title='frecuencia', bargap=0.02)
fig.show()

**Hallazgo:** ~10% con ingreso en 280–5.014 y **gap perfecto** (ni un solo registro entre
5.014 y 280.000). Es error de unidad: esos valores están en miles de pesos.

#### Verificación de la corrección propuesta
El gap limpio prueba que hay dos poblaciones, pero no que el factor sea exactamente 1.000.
Si lo es, al multiplicar el grupo bajo por 1.000 su distribución debe **superponerse** con la
del grupo alto. Un Q-Q plot lo comprueba cuantil a cuantil.

In [ ]:
lo = ing[ing < 50000] * 1000
hi = ing[ing >= 50000]
qs = np.linspace(0.01, 0.99, 99)
q_lo, q_hi = np.quantile(lo, qs), np.quantile(hi, qs)
lim = max(q_lo.max(), q_hi.max())

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Q-Q: grupo "miles" ×1000 vs grupo "pesos"', 'Densidad superpuesta tras la corrección'))
fig.add_scatter(x=q_hi, y=q_lo, mode='markers', marker=dict(color=BICE_AZUL_MED, size=6),
                name='cuantiles', row=1, col=1)
fig.add_scatter(x=[0, lim], y=[0, lim], mode='lines', name='identidad y=x',
                line=dict(color=BICE_ACENTO, dash='dash'), row=1, col=1)
fig.add_histogram(x=hi, nbinsx=60, name='ya en pesos', marker_color=BICE_AZUL, opacity=0.6,
                  histnorm='probability density', row=1, col=2)
fig.add_histogram(x=lo, nbinsx=60, name='corregidos (×1000)', marker_color=BICE_ACENTO, opacity=0.6,
                  histnorm='probability density', row=1, col=2)
fig.update_xaxes(title_text='cuantiles grupo "pesos"', row=1, col=1)
fig.update_yaxes(title_text='cuantiles grupo "miles" ×1000', row=1, col=1)
fig.update_xaxes(title_text='ingreso (CLP)', range=[0, 4e6], row=1, col=2)
fig.update_layout(title='La corrección ×1000 alinea ambas poblaciones', barmode='overlay', height=420)
fig.show()

comp = pd.DataFrame({'cuantil': [0.1, 0.25, 0.5, 0.75, 0.9],
                     'grupo_miles_x1000': np.quantile(lo, [0.1, 0.25, 0.5, 0.75, 0.9]).round(0),
                     'grupo_pesos': np.quantile(hi, [0.1, 0.25, 0.5, 0.75, 0.9]).round(0)})
comp['dif_%'] = ((comp.grupo_miles_x1000 / comp.grupo_pesos - 1) * 100).round(1)
print(comp.to_string(index=False))
print(f"\n% afectado en test: {(test['ingreso_declarado'] < 50000).mean()*100:.1f}%  "
      "← mismo problema, misma corrección")

**Decisión (Opción A):** ×1000 + `flag_ingreso_corregido` (supuesto a validar con negocio).
La diferencia entre cuantiles homólogos tras corregir queda **bajo el 2%**, muy por debajo
del ruido muestral: el factor 1.000 es el correcto.

### 3.5 Punto de masa en edad = 19: ¿código de faltante o clientes jóvenes?

El 5,2% de train tiene exactamente 19 años, seis veces más que a los 20. La sospecha natural
es que 19 sea un **código de relleno** para edad faltante — y si lo fuera, habría que tratarlo
como nulo. Antes de decidir, hay que contrastarlo.

In [ ]:
edad_cnt = train['edad'].value_counts().sort_index()
perfil = pd.DataFrame({
    'grupo': ['edad = 19', 'edad 20-25', 'edad 26+'],
    'antig_laboral_mediana': [
        train.loc[train.edad == 19, 'antiguedad_laboral_meses'].median(),
        train.loc[train.edad.between(20, 25), 'antiguedad_laboral_meses'].median(),
        train.loc[train.edad >= 26, 'antiguedad_laboral_meses'].median()]})

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Distribución de edad — pico en el mínimo', 'Antigüedad laboral: ¿son realmente jóvenes?'))
fig.add_bar(x=edad_cnt.index[:45], y=edad_cnt.values[:45], marker_color=BICE_AZUL,
            showlegend=False, row=1, col=1)
fig.add_annotation(x=19, y=int(edad_cnt.loc[19]), text='19 años<br>5,2% del total',
                   showarrow=True, arrowhead=2, ax=50, ay=-35, row=1, col=1)
fig.add_bar(x=perfil['grupo'], y=perfil['antig_laboral_mediana'], marker_color=BICE_ACENTO,
            text=perfil['antig_laboral_mediana'], textposition='outside', showlegend=False,
            row=1, col=2)
fig.update_xaxes(title_text='edad (años)', row=1, col=1)
fig.update_yaxes(title_text='frecuencia', row=1, col=1)
fig.update_yaxes(title_text='meses (mediana)', row=1, col=2)
fig.update_layout(title='El pico en edad = 19 es coherente con clientes genuinamente jóvenes',
                  height=420)
fig.show()

print(perfil.round(1).to_string(index=False))
print(f"\nPeso del segmento — train: {(train.edad == 19).mean():.2%}  |  "
      f"test: {(test.edad == 19).mean():.2%}")

**Hallazgo:** **no** es un código de faltante. Si lo fuera, el perfil del grupo sería
indistinguible del promedio de la cartera; en cambio su antigüedad laboral mediana es de
**12 meses contra 46** en el tramo 20-25 años. Son personas efectivamente jóvenes, y 19 es
la edad mínima de elegibilidad del producto (censura en el borde, no error).

**Decisión:** no tocar. Imputarlo habría destruido señal legítima.
**Ojo con el dato de abajo:** el segmento pasa de 5,2% en train a 7,9% en test — primera
pista de la deriva que se cuantifica en la sección 8.

### 3.6 Duplicados y categóricas

In [ ]:
cols_sin_id = [c for c in train.columns if c != 'id_solicitud']
n_dup_id      = train['id_solicitud'].duplicated().sum()
n_dup_total   = train.duplicated().sum()
n_dup_sin_id  = train.duplicated(subset=cols_sin_id).sum()

print(f'id_solicitud duplicados          : {n_dup_id}')
print(f'filas duplicadas (con id)        : {n_dup_total}')
print(f'filas duplicadas SIN mirar el id : {n_dup_sin_id}   <-- aquí está el problema')
print()
for c in ['tipo_empleo','region','canal','dia_semana_solicitud']:
    print(f'  {c}: {train[c].nunique()} valores únicos → {sorted(train[c].unique())[:5]}...')

Un `id_solicitud` distinto hace que `df.duplicated()` no detecte nada. Al excluirlo aparecen
**299 pares idénticos en las 19 columnas restantes**. ¿Coincidencia o duplicación de ingesta?
Si fueran solicitudes independientes que casualmente se parecen, coincidirían en unas pocas
columnas, no en todas — y menos con variables continuas de por medio.

In [ ]:
sin_id = train[cols_sin_id]
dup_mask = sin_id.duplicated(keep=False)
dups = train[dup_mask].sort_values(cols_sin_id)
pa, pb = dups.iloc[::2].reset_index(drop=True), dups.iloc[1::2].reset_index(drop=True)
coinc_dup = (pa[cols_sin_id].astype(str).values == pb[cols_sin_id].astype(str).values).sum(axis=1)

rng = np.random.default_rng(0)
ia, ib = rng.choice(len(train), size=(2, len(pa)), replace=True)
coinc_rnd = (train.iloc[ia][cols_sin_id].astype(str).values ==
             train.iloc[ib][cols_sin_id].astype(str).values).sum(axis=1)

fig = go.Figure()
fig.add_histogram(x=coinc_rnd, name='pares aleatorios', marker_color=BICE_AZUL_CLARO,
                  opacity=0.8, xbins=dict(start=-0.5, end=len(cols_sin_id)+0.5, size=1))
fig.add_histogram(x=coinc_dup, name='pares detectados', marker_color=BICE_ALERTA,
                  opacity=0.9, xbins=dict(start=-0.5, end=len(cols_sin_id)+0.5, size=1))
fig.update_layout(
    title='N° de columnas coincidentes: pares duplicados vs pares tomados al azar',
    xaxis_title='columnas idénticas entre las dos filas del par', yaxis_title='n° de pares',
    barmode='overlay', height=400)
fig.show()

print(f'Coincidencia media — duplicados: {coinc_dup.mean():.1f} de {len(cols_sin_id)} columnas')
print(f'Coincidencia media — aleatorios: {coinc_rnd.mean():.1f} de {len(cols_sin_id)} columnas')
print(f"Tasa de default en los duplicados: {train.loc[dup_mask,'default_12m'].mean():.4f} "
      f"(global {train['default_12m'].mean():.4f})")

**Hallazgo (corrige la lectura inicial):** las categóricas están limpias, pero **sí hay
duplicados**: 299 filas repetidas con `id_solicitud` distinto. Los pares detectados coinciden
en las 19 columnas; dos solicitudes al azar coinciden en ~4. Es duplicación de ingesta.

**Decisión:** eliminar las 299 copias (0,66% de train). Mantenerlas duplica el peso de esos
casos y, peor, **filtra entre folds de validación**: la misma solicitud podría quedar en
entrenamiento y en validación a la vez, inflando la métrica.

## 4. Aplicar limpieza (`src/prepare.py`)

In [ ]:
from prepare import preparar_datos
train_clean, test_clean, params = preparar_datos(train, test)
print('Parámetros (de train):', params)
print('Filas train:', len(train), '→', len(train_clean))
print('Nulos restantes:', train_clean[['edad','ingreso_declarado','antiguedad_laboral_meses']].isna().sum().to_dict())
print('Banderas:', [c for c in train_clean.columns if c.startswith('flag_')])

## 5. Análisis de correlaciones ⭐

Primero medimos relaciones lineales entre variables numéricas y con el target.
Este ranking nos dirá **qué variables analizar en detalle** en la sección 6.

In [ ]:
num_cols = ['edad','ingreso_declarado','antiguedad_laboral_meses','antiguedad_cliente_meses',
            'score_buro','deuda_sistema','num_creditos_vigentes','peor_morosidad_12m',
            'num_consultas_buro_3m','num_contactos_ult_trimestre','uso_linea_credito_pct',
            'monto_solicitado','plazo_meses','tasa_interes_anual','ratio_deuda_ingreso',
            'flag_ingreso_corregido','flag_edad_invalida','flag_antiguedad_invalida',
            'flag_ingreso_declarado_faltante','flag_antiguedad_laboral_meses_faltante','default_12m']
corr = train_clean[num_cols].corr()

### 5.1 Tabla completa de correlaciones con el target

In [ ]:
corr_target = corr['default_12m'].drop('default_12m').sort_values(ascending=False)
tabla_corr = corr_target.to_frame('correlacion_con_default')
tabla_corr['direccion'] = np.where(tabla_corr['correlacion_con_default']>=0, '↑ aumenta riesgo', '↓ reduce riesgo')
(tabla_corr.style
   .background_gradient(cmap='RdYlGn_r', subset=['correlacion_con_default'])
   .format({'correlacion_con_default':'{:+.3f}'})
   .set_caption('Correlación de cada variable con default_12m (ordenada)'))

### 5.2 Matriz de correlación interactiva

In [ ]:
fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Matriz de correlación (variables numéricas)')
fig.update_layout(width=900, height=800, xaxis_tickangle=-45, coloraxis_colorbar_title='corr')
fig.update_xaxes(tickfont=dict(size=10)); fig.update_yaxes(tickfont=dict(size=10)); fig.show()

### 5.3 Ranking de correlación con el target

In [ ]:
rank = corr_target.reindex(corr_target.abs().sort_values(ascending=True).index)
colores = [BICE_ALERTA if v>=0 else BICE_OK for v in rank.values]
fig = go.Figure(go.Bar(x=rank.values, y=rank.index, orientation='h', marker_color=colores,
    text=[f'{v:+.3f}' for v in rank.values], textposition='outside'))
fig.update_layout(title='Ranking de correlación con default_12m (por |correlación|)',
    xaxis_title='correlación con el target', yaxis_title='', height=650, margin=dict(l=230))
fig.add_vline(x=0, line_color='#888'); fig.show()

**Nota:** la correlación de Pearson solo aplica a variables numéricas. Para incluir
también las **categóricas** (`tipo_empleo`, `region`, `canal`, `dia_semana`) en la selección
de la sección 6, usamos una medida univariada comparable (AUC) más abajo.

**Alerta que sale de este ranking:** `num_contactos_ult_trimestre` encabeza con una
correlación muy por encima del resto. Una correlación lineal tan alta con un target binario
es anómala y se investiga a fondo en **6.2**.

## 6. Señal predictiva — Top 10 variables

Para rankear **todas** las variables (numéricas y categóricas) con una métrica comparable,
usamos el **AUC univariado** contra el target:
- **Numéricas:** AUC directo del valor (se toma `max(auc, 1-auc)` para capturar dirección).
- **Categóricas:** se codifica cada categoría por su tasa de default (solo train) y se calcula el AUC.

Luego, para el **Top 10**, mostramos la **tasa de default** por *quintil* (numéricas) o por
*categoría* (categóricas/banderas). Esto revela relaciones **no lineales** que la correlación no capta.

In [ ]:
y = train_clean['default_12m']
cat_feats = ['tipo_empleo','region','canal','dia_semana_solicitud']
num_feats = [c for c in num_cols if c != 'default_12m']

strength = {}
for c in num_feats:
    try:
        a = roc_auc_score(y, train_clean[c]); strength[c] = max(a, 1-a)
    except Exception:
        pass
for c in cat_feats:
    rate = train_clean.groupby(c)['default_12m'].transform('mean')
    a = roc_auc_score(y, rate); strength[c] = max(a, 1-a)

rank_auc = pd.Series(strength).sort_values(ascending=False)
top10 = rank_auc.head(10)
print('TOP 10 variables por AUC univariado:')
print(top10.round(3))

fig = px.bar(top10[::-1], orientation='h', color=top10[::-1].values,
             color_continuous_scale=ESCALA_BICE,
             title='Top 10 variables por poder predictivo univariado (AUC)')
fig.update_layout(xaxis_title='AUC univariado', yaxis_title='', height=520,
                  coloraxis_showscale=False, showlegend=False, margin=dict(l=230))
fig.update_xaxes(range=[0.45, 1.0])
fig.show()

### 6.1 Tasa de default por quintil / categoría (Top 10)

Función auxiliar: agrupa por quintil (numéricas con >6 valores) o por categoría, y calcula
la tasa de default y el volumen de cada grupo.

In [ ]:
def señal(var, q=5):
    df = train_clean
    es_cat = (df[var].dtype == object) or (df[var].nunique() <= 6)
    if es_cat:
        g = df.groupby(var)['default_12m'].agg(tasa='mean', n='count').reset_index()
        g[var] = g[var].astype(str); etiqueta = var
    else:
        b = pd.qcut(df[var], q, duplicates='drop')
        g = df.groupby(b, observed=True)['default_12m'].agg(tasa='mean', n='count').reset_index()
        g[var] = g[var].astype(str); etiqueta = f'{var} (quintil)'
    return g, etiqueta, es_cat

top_vars = list(top10.index)
fig = make_subplots(rows=5, cols=2, subplot_titles=top_vars,
                    vertical_spacing=0.06, horizontal_spacing=0.12)
for i, var in enumerate(top_vars):
    g, etiqueta, es_cat = señal(var)
    r, c = i//2 + 1, i%2 + 1
    fig.add_trace(go.Bar(x=g[var], y=g['tasa'], marker_color=BICE_AZUL_MED,
                         showlegend=False, hovertemplate='%{x}<br>tasa %{y:.2%}<extra></extra>'),
                  row=r, col=c)
    fig.add_hline(y=train_clean['default_12m'].mean(), line_dash='dot',
                  line_color=BICE_ACENTO, row=r, col=c)
fig.update_yaxes(tickformat='.0%')
fig.update_xaxes(tickfont=dict(size=8), tickangle=-30)
fig.update_layout(height=1500, title='Tasa de default por quintil / categoría — Top 10 variables')
fig.show()

**Cómo leerlo:** la línea punteada (terracota) es la tasa de default global (~9,9%).
Barras muy por encima/por debajo indican **fuerte poder discriminante**. Un patrón
**monótono** (sube o baja de forma consistente por quintil) es señal limpia y estable;
un patrón en "U" indica no linealidad que los modelos de árboles capturarán mejor que uno lineal.

Todas las variables del Top 10 se comportan como se espera desde el negocio... **salvo la
primera**, cuyo comportamiento es demasiado bueno para ser cierto.

### 6.2 🚨 `num_contactos_ult_trimestre` es una fuga de información

Encabeza el ranking con **AUC 0,954 ella sola** — más que el score de buró y todas las demás
juntas. Ninguna variable disponible al momento de la solicitud puede separar así. El detalle
por valor lo deja en evidencia: con 0 contactos la mora es 0,4%, con 4 contactos es 91%, y
**desde 7 contactos es 100% sin una sola excepción**.

El nombre y el comportamiento apuntan a **gestiones de cobranza registradas después del
desembolso**: no es un predictor, es el resultado con otro nombre.

In [ ]:
g = train_clean.groupby('num_contactos_ult_trimestre')['default_12m'].agg(['size','mean']).reset_index()
auc_leak = roc_auc_score(train_clean['default_12m'], train_clean['num_contactos_ult_trimestre'])

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_bar(x=g['num_contactos_ult_trimestre'], y=g['size'], name='n° de solicitudes',
            marker_color=BICE_AZUL_CLARO, opacity=0.6, secondary_y=True)
fig.add_scatter(x=g['num_contactos_ult_trimestre'], y=g['mean'], mode='lines+markers+text',
                name='tasa de default', line=dict(color=BICE_ALERTA, width=3),
                marker=dict(size=9), text=[f'{v:.0%}' for v in g['mean']],
                textposition='top center', secondary_y=False)
fig.add_hline(y=train_clean['default_12m'].mean(), line_dash='dot', line_color=BICE_AZUL,
              annotation_text='tasa global 9,9%', secondary_y=False)
fig.update_yaxes(title_text='tasa de default', tickformat='.0%', range=[0, 1.15], secondary_y=False)
fig.update_yaxes(title_text='n° de solicitudes', secondary_y=True)
fig.update_layout(title=f'Tasa de default por num_contactos_ult_trimestre · AUC univariado = {auc_leak:.3f}',
                  xaxis_title='num_contactos_ult_trimestre', height=440)
fig.show()

print(g.rename(columns={'size':'n','mean':'tasa_default'}).round(4).to_string(index=False))

#### Confirmación 2: la variable no tiene la misma forma en test
Si fuera una variable legítima de la solicitud, su distribución sería estable entre períodos.
Pero en train llega hasta 12 y **en test se corta en 6**: desaparece justamente la cola que en
train es 100% default. Es el patrón esperable si en train se registró con meses de historia
posterior al desembolso y en test todavía no.

In [ ]:
a = train['num_contactos_ult_trimestre'].value_counts(normalize=True).sort_index()
b = test['num_contactos_ult_trimestre'].value_counts(normalize=True).sort_index()
dist = pd.concat([a.rename('train'), b.rename('test')], axis=1).fillna(0).reset_index(names='contactos')

fig = go.Figure()
fig.add_bar(x=dist['contactos'], y=dist['train'], name='train', marker_color=BICE_AZUL)
fig.add_bar(x=dist['contactos'], y=dist['test'], name='test', marker_color=BICE_ACENTO)
fig.add_vrect(x0=6.5, x1=12.5, fillcolor=BICE_ALERTA, opacity=0.10, line_width=0,
              annotation_text='ausente en test', annotation_position='top right')
fig.update_layout(title=f"Distribución train vs test · máx train = {int(train['num_contactos_ult_trimestre'].max())}, "
                        f"máx test = {int(test['num_contactos_ult_trimestre'].max())}",
                  xaxis_title='num_contactos_ult_trimestre', yaxis_title='proporción',
                  yaxis_tickformat='.0%', barmode='group', height=400)
fig.show()

print(f"P(contactos ≥ 4) — train: {(train['num_contactos_ult_trimestre']>=4).mean():.4f}  |  "
      f"test: {(test['num_contactos_ult_trimestre']>=4).mean():.4f}")

#### Confirmación 3: cuánto costaría usarla
Dos modelos idénticos, mismo split temporal, uno con la variable y otro sin ella. El modelo
con fuga declara un AUC espectacular **que no se podrá reproducir contra el target oculto**.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

tc = train_clean.copy()
tc['fecha_solicitud'] = pd.to_datetime(tc['fecha_solicitud'])
corte = pd.Timestamp('2024-10-01')
tr_in, tr_out = tc[tc.fecha_solicitud < corte], tc[tc.fecha_solicitud >= corte]
feats = num_feats + cat_feats

def _prep(df, cols):
    X = df[cols].copy()
    for c in cat_feats:
        if c in X.columns:
            X[c] = X[c].astype('category')
    return X

res = {}
for nombre, cols in [('CON la variable', feats),
                     ('SIN la variable', [c for c in feats if c != 'num_contactos_ult_trimestre'])]:
    mdl = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06, max_depth=6,
                                         categorical_features='from_dtype', random_state=42)
    mdl.fit(_prep(tr_in, cols), tr_in['default_12m'])
    res[nombre] = roc_auc_score(tr_out['default_12m'], mdl.predict_proba(_prep(tr_out, cols))[:, 1])

fig = go.Figure(go.Bar(x=list(res.keys()), y=list(res.values()),
                       marker_color=[BICE_ALERTA, BICE_OK], width=0.45,
                       text=[f'{v:.3f}' for v in res.values()], textposition='outside'))
fig.update_layout(title='AUC out-of-time (entrena ≤2024-09, valida 2024-10→2025-02)',
                  yaxis_title='AUC', yaxis_range=[0.4, 1.05], height=400, showlegend=False)
fig.show()
for k, v in res.items():
    print(f'{k}: AUC OOT = {v:.4f}')

**Decisión: excluir `num_contactos_ult_trimestre` del modelo.**

Es la decisión más cara en métrica y la más importante en criterio: renunciamos a ~0,14
puntos de AUC declarado a cambio de que la performance reportada sea real. Un modelo que la
incluya diría AUC ≈ 0,98 y se desplomaría contra el target oculto — exactamente la brecha que
el enunciado advierte que se discutirá.

**Acción de negocio:** confirmar con el equipo de Riesgo el momento de registro de esta
variable. Si resultara ser efectivamente pre-solicitud, se reincorpora.

## 7. Análisis del pricing — `tasa_interes_anual` ⭐

La tasa **la asigna el sistema de pricing de Andina** (risk-based pricing): a mayor riesgo
percibido, mayor tasa. Por eso merece un análisis aparte: es predictiva, **pero no es señal
independiente** — codifica el juicio de riesgo que ya hizo otro sistema.

In [ ]:
print('Correlaciones clave de la tasa:')
print(f"  tasa vs score_buro : {train_clean['tasa_interes_anual'].corr(train_clean['score_buro']):+.3f}")
print(f"  tasa vs default    : {train_clean['tasa_interes_anual'].corr(train_clean['default_12m']):+.3f}")
print(f"  score vs default   : {train_clean['score_buro'].corr(train_clean['default_12m']):+.3f}")

### 7.1 Tasa de default por quintil de `tasa_interes_anual`

In [ ]:
b = pd.qcut(train_clean['tasa_interes_anual'], 5)
g = train_clean.groupby(b, observed=True)['default_12m'].agg(tasa='mean', n='count').reset_index()
g['tasa_interes_anual'] = g['tasa_interes_anual'].astype(str)
fig = px.bar(g, x='tasa_interes_anual', y='tasa', text='tasa',
             color='tasa', color_continuous_scale=ESCALA_BICE,
             title='Tasa de default por quintil de tasa_interes_anual')
fig.add_hline(y=train_clean['default_12m'].mean(), line_dash='dot', line_color=BICE_ACENTO,
              annotation_text='tasa global')
fig.update_layout(xaxis_title='quintil de tasa asignada', yaxis_title='tasa de default',
                  coloraxis_showscale=False, yaxis_tickformat='.0%')
fig.update_traces(texttemplate='%{text:.1%}', textposition='outside'); fig.show()

### 7.2 Relación tasa ↔ score (evidencia del pricing basado en riesgo)

Si el pricing se basa en riesgo, debe existir una relación **negativa clara**: peor score → mayor tasa.

In [ ]:
m = train_clean.sample(min(5000, len(train_clean)), random_state=1)
fig = px.scatter(m, x='score_buro', y='tasa_interes_anual',
                 color='default_12m', color_continuous_scale=[BICE_AZUL_CLARO, BICE_ALERTA],
                 opacity=0.5, title='tasa_interes_anual vs score_buro (color = default)')
fig.update_layout(xaxis_title='score_buro', yaxis_title='tasa asignada (%)',
                  coloraxis_colorbar_title='default'); fig.show()

### 7.3 ¿De qué está hecho el pricing?

La tasa correlaciona −0,56 con el score, pero el score no la explica sola. Si el motor de
pricing usa un conjunto de variables que **nosotros también observamos**, entonces la tasa es
en buena medida una función redundante de nuestras propias *features*.

In [ ]:
drivers = ['score_buro','uso_linea_credito_pct','num_consultas_buro_3m','peor_morosidad_12m',
           'deuda_sistema','ratio_deuda_ingreso','num_creditos_vigentes','edad',
           'ingreso_declarado','monto_solicitado','plazo_meses']
cd = train_clean[drivers].corrwith(train_clean['tasa_interes_anual']).sort_values()

fig = go.Figure(go.Bar(x=cd.values, y=cd.index, orientation='h',
                       marker_color=[BICE_ALERTA if v >= 0 else BICE_OK for v in cd.values],
                       text=[f'{v:+.2f}' for v in cd.values], textposition='outside'))
fig.add_vline(x=0, line_color='#888')
fig.update_layout(title='Correlación de cada variable con la tasa asignada — qué mira el motor de pricing',
                  xaxis_title='correlación con tasa_interes_anual', yaxis_title='',
                  height=480, margin=dict(l=210), xaxis_range=[-0.75, 0.6])
fig.show()
print(cd.round(3).to_string())

**Lectura:** el pricing se construye sobre score de buró (−0,56), uso de línea (+0,39),
consultas recientes (+0,38) y morosidad previa (+0,18) — **todas variables que ya tenemos**.
En cambio no mira monto ni plazo (correlación ≈ 0), lo que confirma que es pricing por riesgo
y no por características del producto.

### 7.4 ¿Aporta señal más allá del score?

La prueba decisiva: **dentro de cada decil de score** (es decir, comparando personas con
riesgo de buró equivalente), ¿los que recibieron tasa alta se moran más que los de tasa baja?
Si la respuesta es no, la tasa es puramente derivada y podríamos prescindir de ella.

In [ ]:
tc2 = train_clean.copy()
tc2['dec_score'] = pd.qcut(tc2['score_buro'], 10, labels=False, duplicates='drop')
tc2['tasa_alta'] = tc2.groupby('dec_score')['tasa_interes_anual'].transform(lambda s: s > s.median())
gg = tc2.groupby(['dec_score','tasa_alta'])['default_12m'].mean().unstack()

fig = go.Figure()
fig.add_bar(x=gg.index, y=gg[False], name='tasa BAJA dentro del decil', marker_color=BICE_AZUL_CLARO)
fig.add_bar(x=gg.index, y=gg[True], name='tasa ALTA dentro del decil', marker_color=BICE_ACENTO)
fig.update_layout(title='Tasa de default por decil de score, separando por tasa alta/baja dentro del decil',
                  xaxis_title='decil de score_buro (0 = peor score)', yaxis_title='tasa de default',
                  yaxis_tickformat='.0%', barmode='group', height=430)
fig.show()

# Cuánta señal queda tras remover linealmente el efecto del score
z = np.polyfit(train_clean['score_buro'], train_clean['tasa_interes_anual'], 1)
resid = train_clean['tasa_interes_anual'] - np.polyval(z, train_clean['score_buro'])
print(f"AUC de la tasa sola                      : {roc_auc_score(y, train_clean['tasa_interes_anual']):.4f}")
print(f"AUC del residuo de la tasa (sin el score): {roc_auc_score(y, resid):.4f}")
print(f"Lift medio tasa alta / tasa baja dentro del decil: {(gg[True]/gg[False]).mean():.2f}x")

**Hallazgos del pricing:**
- La tasa correlaciona **≈ −0,56 con el score** → confirma **pricing basado en riesgo**.
- Correlaciona **≈ +0,21 con el default**: es predictiva, pero de forma **derivada** (vía score).
- Dentro de cada decil de score, los de tasa alta se moran **~1,28× más** que los de tasa
  baja, y el patrón se repite en los **10 deciles**. Hay señal residual, pero es modesta: el
  AUC del residuo cae de 0,706 a **0,533**, apenas sobre el azar. La tasa aporta poco que las
  demás variables no aporten ya.
- **Implicancia de modelado:** entrenaremos **con y sin** `tasa_interes_anual`. Si el modelo
  depende demasiado de ella, se degradará cuando cambie la política de pricing. Reportar ambos
  demuestra criterio y protege el rendimiento en producción. Dado que el residuo es de 0,533,
  la apuesta es que el costo de excluirla será bajo.

## 8. Análisis temporal (define la validación)

In [ ]:
train_clean['mes'] = pd.to_datetime(train_clean['fecha_solicitud']).dt.to_period('M').astype(str)
test_clean['mes']  = pd.to_datetime(test_clean['fecha_solicitud']).dt.to_period('M').astype(str)

mensual = train_clean.groupby('mes')['default_12m'].agg(['size','mean']).reset_index()
mensual.columns = ['mes','n','tasa']
se = np.sqrt(mensual.tasa*(1-mensual.tasa)/mensual.n)
z2 = np.polyfit(np.arange(len(mensual)), mensual.tasa, 1)
meses_test = sorted(test_clean['mes'].unique())
proy = np.polyval(z2, np.arange(len(mensual), len(mensual)+len(meses_test)))

fig = go.Figure()
fig.add_scatter(x=mensual['mes'], y=mensual.tasa+1.96*se, mode='lines', line=dict(width=0),
                showlegend=False, hoverinfo='skip')
fig.add_scatter(x=mensual['mes'], y=mensual.tasa-1.96*se, mode='lines', line=dict(width=0),
                fill='tonexty', fillcolor='rgba(169,189,223,0.45)', name='IC 95%', hoverinfo='skip')
fig.add_scatter(x=mensual['mes'], y=mensual.tasa, mode='lines+markers', name='tasa observada',
                line=dict(color=BICE_AZUL, width=3), marker=dict(size=8))
fig.add_scatter(x=meses_test, y=proy, mode='lines+markers', name='extrapolación (período test)',
                line=dict(color=BICE_ACENTO, dash='dash', width=2))
fig.add_hline(y=train_clean['default_12m'].mean(), line_dash='dot', line_color=BICE_PETROLEO,
              annotation_text='promedio train 9,9%', annotation_position='bottom left')
fig.update_layout(title='Tasa de default por mes de solicitud — y proyección al período de test',
                  xaxis_title='mes', yaxis_title='tasa de default', xaxis_tickangle=-45,
                  yaxis_tickformat='.0%', height=440)
fig.show()

print(mensual.assign(tasa=lambda d: (d.tasa*100).round(2)).to_string(index=False))
print(f'\nPendiente: +{z2[0]*100:.2f} pp por mes  |  tasa proyectada para test: {proy.mean():.1%}')

**Hallazgo CRÍTICO:** el default **crece** en el tiempo (7% → 13%). Un split aleatorio sería
optimista y deshonesto → usaremos **validación temporal (out-of-time)**.

**Consecuencia adicional:** el 9,9% promedio de train **no** es la tasa esperada del período
de test. Extrapolando la tendencia, el test debería estar en torno al **14%**. Como la
política de aprobación compara probabilidades contra umbrales absolutos (sección 9), esto
obliga a **recalibrar** el modelo hacia la tasa esperada, no la histórica.

**Nota sobre madurez del target:** con una ventana de 12 meses, cabía sospechar que las
cohortes recientes de train tuvieran etiquetas incompletas, lo que produciría una **caída**
artificial en los últimos meses. El gráfico muestra lo contrario: la tasa sube hasta el final.
La data **no** tiene ese problema y el deterioro es real.

### 8.1 ¿Qué explica el deterioro? Cambio de mezcla de originación

In [ ]:
mix_tr = train_clean.groupby('mes').agg(digital=('canal', lambda s: (s=='digital').mean()),
                                        score=('score_buro','mean'), edad=('edad','mean')).reset_index()
mix_te = test_clean.groupby('mes').agg(digital=('canal', lambda s: (s=='digital').mean()),
                                       score=('score_buro','mean'), edad=('edad','mean')).reset_index()

fig = make_subplots(rows=1, cols=3, subplot_titles=('% canal digital','score de buró promedio','edad promedio'))
for i, col in enumerate(['digital','score','edad'], start=1):
    fig.add_scatter(x=mix_tr['mes'], y=mix_tr[col], mode='lines+markers', name='train',
                    line=dict(color=BICE_AZUL, width=2.5), showlegend=(i==1), row=1, col=i)
    fig.add_scatter(x=mix_te['mes'], y=mix_te[col], mode='lines+markers', name='test',
                    line=dict(color=BICE_ACENTO, width=2.5, dash='dot'), showlegend=(i==1), row=1, col=i)
fig.update_yaxes(tickformat='.0%', row=1, col=1)
fig.update_xaxes(tickangle=-45, tickfont=dict(size=8))
fig.update_layout(title='Deriva de la mezcla de originación entre train y test', height=420)
fig.show()

print('canal — train:', train_clean.canal.value_counts(normalize=True).round(3).to_dict())
print('canal — test :', test_clean.canal.value_counts(normalize=True).round(3).to_dict())
print('\ntasa de default por canal (train):')
print(train_clean.groupby('canal')['default_12m'].agg(['size','mean']).round(4).to_string())

**Hallazgo:** el canal digital pasa de **36% a 69%** del volumen entre el inicio de train y el
final de test, y tiene **12,5% de default frente a 7,2% en sucursal**. El deterioro no es un
shock macro: es un cambio de mezcla de originación. En paralelo el perfil se vuelve más joven
y de menor score.

### 8.2 Cuantificar la deriva: validación adversarial

Entrenamos un clasificador que intenta distinguir "¿esta fila viene de train o de test?".
Si ambas poblaciones fueran intercambiables el AUC sería 0,5.

**Qué es y qué no es esta prueba.** Es un **diagnóstico de magnitud**, no un criterio para
eliminar variables. Que una variable derive en el tiempo no la invalida: `canal` y `edad`
derivan fuertemente (sección 8.1) y ambas son información legítima y valiosa sobre cómo está
cambiando el perfil de quien solicita crédito. Eliminarlas por derivar sería tirar justamente
la señal que explica el deterioro de la cartera.

El único caso en que la deriva sí justifica excluir una variable es cuando además hay otra
razón de fondo — como en `num_contactos_ult_trimestre`, que se excluye por ser una fuga
(sección 6.2), no por derivar. Aquí medimos cuánto de la deriva total explica esa exclusión.

In [ ]:
from sklearn.model_selection import cross_val_predict

f_adv = [c for c in num_feats if c in test_clean.columns] + cat_feats
A = pd.concat([_prep(train_clean, f_adv), _prep(test_clean, f_adv)], ignore_index=True)
y_adv = np.r_[np.zeros(len(train_clean)), np.ones(len(test_clean))]

adv = {}
for nombre, drop in [('todas las variables', []),
                     ('sin la variable con fuga', ['num_contactos_ult_trimestre'])]:
    Xa = A.drop(columns=[c for c in drop if c in A.columns])
    p = cross_val_predict(HistGradientBoostingClassifier(max_iter=120, categorical_features='from_dtype',
                                                        random_state=0),
                          Xa, y_adv, cv=3, method='predict_proba')[:, 1]
    adv[nombre] = roc_auc_score(y_adv, p)

fig = go.Figure(go.Bar(x=list(adv.keys()), y=list(adv.values()),
                       marker_color=[BICE_ALERTA, BICE_ACENTO], width=0.4,
                       text=[f'{v:.3f}' for v in adv.values()], textposition='outside'))
fig.add_hline(y=0.5, line_dash='dash', line_color=BICE_PETROLEO,
              annotation_text='0,5 = train y test indistinguibles (ideal)')
fig.update_layout(title='Validación adversarial: ¿cuán distinto es test respecto de train?',
                  yaxis_title='AUC adversarial', yaxis_range=[0.45, 0.72], height=420, showlegend=False)
fig.show()
for k, v in adv.items():
    print(f'{k:>28}: AUC = {v:.4f}')

**Hallazgo:** el AUC adversarial baja de 0,65 a ~0,60 al excluir la variable con fuga: una
parte relevante de la "diferencia" entre train y test era en realidad el artefacto de esa
columna, no un cambio genuino de la población.

El 0,60 restante es **deriva real y legítima**: la mezcla de canal, la edad y el score se
mueven de verdad entre ambos períodos. Eso no se corrige eliminando variables, se maneja con
el diseño de validación (sección 8.3) y con la recalibración hacia la tasa esperada del
período de test. La acción que sí corresponde es **monitorear** estas variables en producción:
si la mezcla de canal sigue moviéndose, el modelo habrá que reentrenarlo antes de lo previsto.

### 8.3 Esquema de validación: backtesting con ventanas expansivas

Un split temporal único entregaría una sola estimación, sin noción de su varianza, y dejaría
fuera del entrenamiento los meses más recientes (los más parecidos al test). Usamos cuatro
folds sucesivos que imitan cómo se usaría el modelo en producción: entrenar con el pasado,
validar con el futuro inmediato.

In [ ]:
meses = sorted(train_clean['mes'].unique())
pos = {m: i for i, m in enumerate(meses)}
cortes = [('2024-07','2024-09'), ('2024-09','2024-11'), ('2024-11','2025-01'), ('2025-01','2025-02')]

fig = go.Figure()
for i, (fin_tr, fin_va) in enumerate(cortes):
    ini_va = meses[pos[fin_tr]+1]
    fig.add_shape(type='rect', x0=-0.5, x1=pos[fin_tr]+0.5, y0=i-0.35, y1=i+0.35,
                  fillcolor=BICE_AZUL, opacity=0.85, line_width=0)
    fig.add_shape(type='rect', x0=pos[ini_va]-0.5, x1=pos[fin_va]+0.5, y0=i-0.35, y1=i+0.35,
                  fillcolor=BICE_ACENTO, opacity=0.9, line_width=0)
fig.add_shape(type='rect', x0=len(meses)-0.5, x1=len(meses)+4.5, y0=-0.35, y1=len(cortes)-0.65,
              fillcolor=BICE_ALERTA, opacity=0.12, line_width=0)
fig.add_annotation(x=len(meses)+2, y=len(cortes)-1.1, text='período de TEST<br>(sin target)',
                   showarrow=False, font=dict(color=BICE_ALERTA, size=12))
fig.update_layout(title='Backtesting temporal · azul = entrenamiento, terracota = validación',
    xaxis=dict(tickmode='array', tickvals=list(range(len(meses)+5)),
               ticktext=meses + meses_test, tickangle=-45),
    yaxis=dict(tickmode='array', tickvals=list(range(len(cortes))),
               ticktext=[f'fold {i+1}' for i in range(len(cortes))], autorange='reversed'),
    height=360, showlegend=False)
fig.show()

**Decisión:** reportar como performance esperada el promedio de los folds, dando peso especial
al **último** (el más cercano al período de test). Cualquier métrica de k-fold aleatorio se
reporta solo como referencia, nunca como la estimación oficial.

## 9. Traducción a la política de aprobación

El modelo entrega una probabilidad `p`. Eso **todavía no es una decisión**. La decisión sale de
comparar lo que se gana si el crédito se paga contra lo que se pierde si cae en default.
Esta sección hace esa traducción en tres pasos: primero con un caso concreto, después con la
fórmula general, y al final midiendo cuánta plata vale la política.

### 9.1 Un caso concreto

Solicitud de **$2.000.000 a 24 meses**. Los tres escenarios posibles:

| Escenario | Resultado |
|---|---|
| El cliente **paga** | `2.000.000 × 12% × (24/12) × 0,5` = **+$240.000** |
| El cliente cae en **default** | `2.000.000 × 55%` = **−$1.100.000** |
| Se **rechaza** la solicitud | **$0** |

Aprobar conviene mientras el valor esperado sea positivo. El punto de equilibrio es donde la
ganancia esperada iguala a la pérdida esperada:

$$(1-p)\cdot 240.000 = p\cdot 1.100.000 \;\Longrightarrow\; p^* = \frac{240.000}{1.340.000} = 17{,}9\%$$

**A ese cliente se le aprueba mientras su probabilidad de default esté bajo 17,9%.**

Lo que manda es la asimetría: se pierde **4,6 veces más** de lo que se gana, así que la
inmensa mayoría tiene que pagar para que el negocio funcione.

### 9.2 El umbral depende del plazo, y solo del plazo

Generalizando el ejemplo, con $G$ = ganancia si paga y $L$ = pérdida si cae:

$$(1-p)\cdot G > p\cdot L \quad\Longrightarrow\quad p^* = \frac{G}{G+L}$$

Reemplazando las fórmulas del enunciado:

$$G = \text{monto}\times 0{,}12\times\tfrac{\text{plazo}}{12}\times 0{,}5 = \text{monto}\times 0{,}005\times\text{plazo}
\qquad L = \text{monto}\times 0{,}55$$

$$p^* = \frac{\text{monto}\times 0{,}005\times\text{plazo}}{\text{monto}\times 0{,}005\times\text{plazo} + \text{monto}\times 0{,}55}
= \frac{0{,}005\times\text{plazo}}{0{,}005\times\text{plazo} + 0{,}55}$$

**El monto aparece en el numerador y en el denominador, así que se cancela.** Este resultado
es contraintuitivo y conviene decirlo explícito en el informe:

> El monto **no cambia a quién se aprueba**, solo cuánto se gana o se pierde con esa decisión.
> Duplicar el monto duplica la ganancia y duplica la pérdida en la misma proporción, así que
> el punto de equilibrio no se mueve.

El **plazo** sí lo mueve, porque los intereses se acumulan con el tiempo mientras la pérdida
por incumplimiento no depende del plazo. A un cliente riesgoso que pide a 48 meses se le puede
decir que sí; al mismo cliente pidiendo a 6 meses, no.

In [ ]:
plazos = np.arange(6, 61)
umbral = (0.005*plazos)/(0.005*plazos + 0.55)
pr = sorted(train_clean['plazo_meses'].unique())
ur = (0.005*np.array(pr))/(0.005*np.array(pr) + 0.55)
vol = train_clean['plazo_meses'].value_counts(normalize=True).sort_index()

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_bar(x=vol.index, y=vol.values, name='% de solicitudes', marker_color=BICE_AZUL_CLARO,
            opacity=0.6, secondary_y=True)
fig.add_scatter(x=plazos, y=umbral, mode='lines', name='umbral óptimo p*',
                line=dict(color=BICE_AZUL, width=3), secondary_y=False)
fig.add_scatter(x=pr, y=ur, mode='markers+text', name='plazos presentes en la data',
                marker=dict(color=BICE_ACENTO, size=11), text=[f'{u:.1%}' for u in ur],
                textposition='top center', secondary_y=False)
fig.update_yaxes(title_text='umbral de aprobación p*', tickformat='.0%', range=[0, 0.4], secondary_y=False)
fig.update_yaxes(title_text='% de solicitudes', tickformat='.0%', secondary_y=True)
fig.update_layout(title='Umbral de aprobación óptimo por plazo — el monto no interviene',
                  xaxis_title='plazo (meses)', height=440)
fig.show()

print(pd.DataFrame({'plazo_meses': pr, 'umbral_optimo_%': (ur*100).round(1),
                    'pct_solicitudes': (vol.reindex(pr).values*100).round(1)}).to_string(index=False))

**Hallazgo:** el umbral va de **5,2% a 6 meses** hasta **30,4% a 48 meses**. La regla de
decisión recomendada es simplemente: *aprobar si `p` está bajo el umbral que corresponde al
plazo solicitado*.

### 9.3 ¿Cuánto vale esta política? Backtest con la economía real

Hasta aquí todo es aritmética. Falta lo que pide el enunciado: **cuánta plata gana la política
frente a aprobar todo**. Para eso hay que simular la decisión sobre datos con resultado
conocido.

#### Diseño del experimento

Tres bloques que respetan el orden temporal:

| Bloque | Período | Para qué |
|---|---|---|
| **Entrenamiento** | hasta 2024-08 | ajustar el modelo |
| **Calibración** | 2024-09 a 2024-10 | corregir el nivel de las probabilidades y elegir el umbral global de comparación |
| **Evaluación** | 2024-11 a 2025-02 | medir la ganancia, sobre datos que el modelo nunca vio |

El umbral global se elige en el bloque de calibración, **no** en el de evaluación. Barrerlo
sobre los mismos datos donde después se mide produciría una ganancia inflada que no se
repetiría en producción.

#### Por qué estos dos algoritmos

**`HistGradientBoostingClassifier` — el modelo.**
Es la implementación de *gradient boosting* sobre árboles que viene incluida en scikit-learn,
conceptualmente equivalente a LightGBM (de hecho está inspirada en él: agrupa los valores de
cada variable en histogramas para encontrar los cortes rápido). Se eligió por tres razones:

- **Árboles y no una regresión logística**, porque las relaciones que documentó la sección 6.1
  no son lineales, hay categóricas con muchos niveles (`region` tiene 16) y el algoritmo maneja
  valores faltantes de forma nativa, sin obligar a decidir la imputación de antemano.
- **Scikit-learn y no LightGBM**, porque el EDA no debería depender de la librería del modelo
  final. Aquí no se busca el mejor modelo posible: se busca un **diagnóstico comparable** entre
  escenarios. Lo que importa es que sea *exactamente el mismo modelo* a ambos lados de cada
  comparación (con y sin fuga, política contra aprobar todo).
- **Es un punto de partida, no el entregable.** El modelo definitivo del paso 2 será LightGBM
  con búsqueda de hiperparámetros. Las cifras de esta sección son un piso razonable de lo que
  se puede esperar, no el techo.

**`IsotonicRegression` — la calibración.**
El problema a resolver: el modelo **ordena bien** a los clientes pero sus probabilidades están
corridas hacia abajo, porque entrenó en un período con menos mora que aquel donde se aplica.

- **Qué hace:** aprende una función que traduce "probabilidad predicha" → "probabilidad
  realmente observada", ajustada sobre el bloque de calibración, que el modelo no vio al
  entrenar. Si el modelo dice 8% donde en realidad ocurre 11%, la función corrige ese 8% a 11%.
- **Por qué "isotónica" (monótona creciente):** al ser monótona **preserva el orden** de los
  clientes. Solo reubica los valores, nunca cambia quién es más riesgoso que quién. Por eso el
  AUC queda prácticamente idéntico y lo que mejora es el nivel — que es justo lo que necesita
  una política basada en umbrales absolutos.
- **Por qué isotónica y no Platt (sigmoide):** Platt asume que la distorsión tiene forma de S y
  la ajusta con dos parámetros. La isotónica no asume ninguna forma: se adapta a cualquier
  distorsión mientras sea monótona. Con ~6.800 casos en el bloque de calibración hay datos de
  sobra; con pocos datos la isotónica sobreajusta y Platt sería la elección correcta.

In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

tc3 = train_clean.copy()
tc3['fecha'] = pd.to_datetime(tc3['fecha_solicitud'])
EXCLUIR = ['id_solicitud', 'fecha_solicitud', 'fecha', 'mes', 'default_12m',
           'num_contactos_ult_trimestre']          # <- la fuga queda fuera
FEATS = [c for c in tc3.columns if c not in EXCLUIR]

ent = tc3[tc3.fecha <  '2024-09-01']
cal = tc3[(tc3.fecha >= '2024-09-01') & (tc3.fecha < '2024-11-01')]
evl = tc3[tc3.fecha >= '2024-11-01']
print(f'entrena {len(ent):,} (→2024-08)  |  calibra {len(cal):,} (09-10)  |  evalúa {len(evl):,} (11→02)')

mdl = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=6,
                                     categorical_features='from_dtype', random_state=42)
mdl.fit(_prep(ent, FEATS), ent['default_12m'])
p_cal = mdl.predict_proba(_prep(cal, FEATS))[:, 1]
p_evl = mdl.predict_proba(_prep(evl, FEATS))[:, 1]

iso = IsotonicRegression(out_of_bounds='clip').fit(p_cal, cal['default_12m'])
p_evl_cal = iso.predict(p_evl)

print(f"\n                 AUC     Brier    prob. media predicha")
print(f"sin calibrar   {roc_auc_score(evl.default_12m, p_evl):.4f}  {brier_score_loss(evl.default_12m, p_evl):.5f}   {p_evl.mean():.1%}")
print(f"con calibrar   {roc_auc_score(evl.default_12m, p_evl_cal):.4f}  {brier_score_loss(evl.default_12m, p_evl_cal):.5f}   {p_evl_cal.mean():.1%}")
print(f"\ntasa de default REAL en el bloque de evaluación: {evl.default_12m.mean():.1%}")

Nótese lo que acaba de pasar: **el AUC no se mueve** (0,833 vs 0,832) pero la probabilidad
media predicha pasa de 9,5% a 11,7%, acercándose al 12,2% real. La calibración no mejoró el
ordenamiento — no era su trabajo — sino el **nivel**, que es lo que la política necesita.

#### Resultado: ganancia de cada política

In [ ]:
MM = 1e6
d = evl.copy()
d['p'] = p_evl_cal
d['G'] = d['monto_solicitado'] * 0.005 * d['plazo_meses']       # ganancia si paga
d['L'] = d['monto_solicitado'] * 0.55                            # pérdida si cae
d['margen'] = np.where(d['default_12m'] == 1, -d['L'], d['G'])   # resultado real si se aprueba
d['umbral_plazo'] = d['G'] / (d['G'] + d['L'])

# Umbral global de comparación: se elige en el bloque de calibración
dc = cal.copy(); dc['p'] = iso.predict(p_cal)
Gc = dc['monto_solicitado'] * 0.005 * dc['plazo_meses']; Lc = dc['monto_solicitado'] * 0.55
dc['margen'] = np.where(dc['default_12m'] == 1, -Lc, Gc)
u_global = max(((u, dc.loc[dc.p < u, 'margen'].sum()) for u in np.arange(0.02, 0.60, 0.005)),
               key=lambda t: t[1])[0]

politicas = {
    'Aprobar todo\n(situación base)':          (d['margen'].sum()/MM, 1.0),
    f'Umbral global único\n({u_global:.0%})':  (d.loc[d.p < u_global, 'margen'].sum()/MM, (d.p < u_global).mean()),
    'Umbral por plazo\n(recomendada)':         (d.loc[d.p < d.umbral_plazo, 'margen'].sum()/MM, (d.p < d.umbral_plazo).mean()),
    'Oráculo\n(si supiéramos el futuro)':      (d.loc[d.default_12m == 0, 'margen'].sum()/MM, (d.default_12m == 0).mean()),
}
nombres = list(politicas); valores = [v[0] for v in politicas.values()]; aprob = [v[1] for v in politicas.values()]

fig = go.Figure(go.Bar(
    x=nombres, y=valores, marker_color=[BICE_AZUL_CLARO, BICE_AZUL_MED, BICE_ACENTO, BICE_PETROLEO],
    text=[f'{v:,.0f} MM<br><span style="font-size:11px">aprueba {a:.0%}</span>' for v, a in zip(valores, aprob)],
    textposition='outside'))
fig.add_annotation(x=0.5, y=max(valores)*0.60, showarrow=False,
                   text=f'<b>+{valores[2]-valores[0]:,.0f} MM</b><br>({(valores[2]/valores[0]-1)*100:.0f}% sobre la base)',
                   font=dict(color=BICE_ALERTA, size=13))
fig.update_layout(title='Ganancia de la cartera según política · backtest 2024-11 → 2025-02',
                  yaxis_title='ganancia (millones de CLP)', height=470, showlegend=False,
                  yaxis_range=[0, max(valores)*1.25])
fig.show()

for n, (v, a) in politicas.items():
    print(f"{n.splitlines()[0]:<22}: {v:>9,.0f} MM CLP  |  aprueba {a:.1%}")

**El titular es la primera barra contra la tercera.** Pasar de aprobar todo a aplicar la
política multiplica la ganancia por más de tres: de 337 a ~1.057 millones.

Dos matices honestos que deben quedar en el informe:

- El salto grande viene de **tener un umbral**, no del refinamiento por plazo. El umbral por
  plazo aporta sobre el corte único, pero solo ~2-3%.
- **Aprobar todo apenas es rentable.** Con 12% de mora y 55% de LGD, la política actual de
  reglas simples está al borde de destruir valor — que es exactamente el síntoma que motivó
  este encargo.

### 9.4 El menú para negociar: ganancia versus tasa de aprobación

El óptimo económico asume capital ilimitado y ninguna meta de volumen. En la práctica Riesgo y
Comercial negocian ese punto, así que conviene mostrar el menú completo: para cada tasa de
aprobación, cuánta ganancia se captura.

In [ ]:
grid = np.arange(0.01, 0.75, 0.0025)
curva = pd.DataFrame({'aprobacion': [(d.p < u).mean() for u in grid],
                      'ganancia':   [d.loc[d.p < u, 'margen'].sum()/MM for u in grid]})
g_plazo, a_plazo = politicas['Umbral por plazo\n(recomendada)']
g_base = politicas['Aprobar todo\n(situación base)'][0]

fig = go.Figure()
fig.add_scatter(x=curva.aprobacion, y=curva.ganancia, mode='lines',
                line=dict(color=BICE_AZUL, width=3), name='mejor corte global posible')
fig.add_scatter(x=[a_plazo], y=[g_plazo], mode='markers+text', name='política recomendada',
                marker=dict(color=BICE_ACENTO, size=16, symbol='star'),
                text=[f'  recomendada: {a_plazo:.0%} · {g_plazo:,.0f} MM'], textposition='bottom right')
fig.add_scatter(x=[1.0], y=[g_base], mode='markers+text', name='situación base',
                marker=dict(color=BICE_ALERTA, size=13), text=['  aprobar todo'], textposition='middle left')
fig.add_hline(y=g_base, line_dash='dot', line_color=BICE_ALERTA,
              annotation_text='nivel actual', annotation_position='bottom left')
fig.add_vrect(x0=0.65, x1=0.85, fillcolor=BICE_AZUL_CLARO, opacity=0.20, line_width=0,
              annotation_text='zona plana', annotation_position='top left')
fig.update_layout(title='Ganancia total según tasa de aprobación',
                  xaxis_title='tasa de aprobación', yaxis_title='ganancia (millones de CLP)',
                  xaxis_tickformat='.0%', height=470)
fig.show()

print('Ganancia a distintas metas de aprobación:')
for meta in [0.60, 0.70, 0.80, 0.90, 1.00]:
    fila = curva.iloc[(curva.aprobacion - meta).abs().argmin()]
    print(f'  aprobar ~{meta:.0%}: {fila.ganancia:>8,.0f} MM')

**Dos lecturas de este gráfico:**

1. **La estrella queda por encima de la curva.** La curva recorre *todos* los cortes globales
   posibles, así que su máximo es el techo de cualquier umbral único. Que la política por plazo
   lo supere es la prueba visual de que diferenciar por plazo aporta algo real, aunque sea poco.
2. **La curva es plana entre ~65% y ~85% de aprobación.** Buena noticia operativa: la ganancia
   no depende de acertar el corte al decimal, así que hay margen para acomodar metas
   comerciales sin destruir valor. Lo caro es quedarse en el extremo derecho (aprobar todo).

### 9.5 Limitaciones de esta estimación

Tres advertencias que deben ir en el informe ejecutivo:

1. **El backtest usa el target real; `test.csv` no lo tiene.** La ganancia que se reporte sobre
   test es una *estimación*, bajo el supuesto de que el modelo rinda allí como en el bloque de
   evaluación. Dado el deterioro documentado en la sección 8, es más probable que rinda algo
   peor que mejor.
2. **Sesgo de selección (*reject inference*).** Todas las filas son créditos **desembolsados**:
   ya pasaron el filtro de reglas vigente. El modelo no observa el comportamiento de quienes hoy
   se rechazan. La estimación es válida para la población que hoy llega a desembolso, no para el
   universo completo de solicitantes.
3. **El óptimo asume capital ilimitado y ninguna meta de volumen.** Si existe una restricción
   comercial de aprobación mínima, el corte deja de ser económico y pasa a ser operativo: para
   eso está la curva de la sección 9.4.

## Resumen de decisiones (Paso 1)

| # | Problema | Evidencia | Decisión |
|---|----------|-----------|----------|
| 1 | Nulos ingreso (18%) / antigüedad (16%), estables train-test | §3.1 | Imputar mediana (train) + bandera |
| 2 | Edad ≥ 100 (70 filas, hasta 133 años) | §3.2 | → nulo → imputar + `flag_edad_invalida` |
| 3 | Antigüedad laboral > vida laboral posible (46 filas) | §3.3 | → nulo → imputar + `flag_antiguedad_invalida` |
| 4 | Ingreso en miles (~10%, gap limpio, Q-Q confirma ×1000) | §3.4 | ×1000 + bandera (Opción A) |
| 5 | Pico en edad = 19 (5,2%) | §3.5 | **No tocar**: censura de elegibilidad, no faltante |
| 6 | 299 duplicados exactos con `id` distinto | §3.6 | Eliminar copias (evita fuga entre folds) |
| 7 | Desbalance del target (9,9%) | §2 | Métricas AUC / KS / PR-AUC |
| 8 | **`num_contactos_ult_trimestre` es fuga: AUC 0,954, 100% de mora en la cola, ausente en test** | **§6.2** | **Excluir del modelo; confirmar con Riesgo** |
| 9 | `tasa` refleja pricing por riesgo (residuo AUC 0,533) | §7.3-7.4 | Modelar con y sin la variable |
| 10 | Default creciente 7% → 13%; canal digital 36% → 69% | §8, §8.1 | Validación out-of-time; recalibrar a ~14% |
| 11 | Deriva adversarial 0,65 → 0,60 al excluir la fuga; el resto es deriva real | §8.2 | **No** eliminar variables por derivar; monitorear canal en producción |
| 12 | Umbral óptimo depende del plazo (9,8% → 30,4%), no del monto | §9, §9.1 | Política de umbral por plazo |
| 13 | La política triplica la ganancia vs aprobar todo (337 → ~1.057 MM); el umbral por plazo aporta ~2-3% extra | §9.2, §9.3 | Recomendar la política; presentar la curva ganancia-aprobación |
| 14 | Sin calibrar, el modelo subestima la mora (9,5% vs 12,2% real) y aprueba 6 pp de más; el AUC no lo detecta | §9.4 | Calibración isotónica + reportar Brier |

**Anti-leakage:** todos los estadísticos se calculan solo en train y se aplican a test.

**Próximo paso:** pipeline de modelado con LightGBM sobre el set sin fuga, calibración sobre
el fold out-of-time más reciente y cuantificación de la ganancia esperada de la política
versus aprobar todo.